Below is a backend blueprint for **TrackField CRM** that matches the frontend you already built.

Your frontend currently has three main app areas: **Super Admin**, **Company Admin**, and **User Dashboard**, with modules like companies, users, employees, leads, deals, contacts, calendar, payments, auto([GitHub][1])e control, settings, tasks, tickets, and activity log. The README also confirms dynamic module visibility, role-permission matrix, notifications, and calendar behavior, which means the backend must support **multi-tenant isolation**, **per-company feature flags**, **per-role action permissions**, and **role-aware data filtering**. ([GitHub][1])1) Frontend analysis

### Pages

From your current frontend structure, these are the backend-facing pages/modules:

**Public**

* Login

**Super Admin**

* Dashboard
* Companies
* Users
* Plans
* Analytics
* Activity Log

**Company Admin**

* Dashboard
* Employees
* Leads
* Deals
* Contacts
* Calendar
* Payments
* Automation
* Reports
* Module Control
* Settings

**Employee/User**

* Dashboard
* My Leads
* My Deals
* Tasks
* Contacts
* Tickets

This maps directly to the repo README structure and role architecture. ([GitHub][1]) Flow

Your frontend flow is essentially:

**Login**

* User logs in
* Backend returns JWT access token, refresh token, user profile, company context, role, allowed modules, and permissions

**App bootstrap**

* Frontend loads:

  * current user
  * current company
  * visible modules
  * permission matrix
  * notification count

**Super Admin flow**

* manages platform-wide companies, subscriptions, global users, analytics, audit logs

**Company Admin flow**

* manages company users/employees
* toggles modules for that company
* configures role permissions
* manages leads, deals, contacts, calendar, payments, automation, reports, settings

**Employee flow**

* only sees modules enabled by company
* only sees actions allowed by role
* mostly works on assigned leads, deals, tasks, contacts, tickets

This is exactly the kind of flow that needs backend enforcement, not just frontend hiding, because module hiding and permission matrix in your app are core platform behavior. ([GitHub][1]) Required APIs from frontend perspective

At minimum, your frontend needs these API groups:

* auth
* me/session bootstrap
* companies
* subscriptions/plans
* users/employees
* roles & permissions
* module control
* dashboard/analytics/reports
* leads
* deals
* contacts
* tasks
* calendar/events
* payments/invoices
* notifications
* tickets
* automation/workflows
* activity logs
* settings

The repo explicitly says mock CRUD in `AuthContext.jsx` should map to real API calls and JWT auth with bearer token headers, so this backend should be built around that contract. ([GitHub][1])

# 2) Scalable backend architecture

## Recommended stack

* **Node.js**
* **Express.js**
* **MongoDB + Mongoose**
* **Redis** for cache, rate-limit counters, queues, sessions if needed
* **BullMQ** for background jobs
* **JWT** access token + refresh token
* **Zod** or **Joi** for validation
* **Winston/Pino** for logging
* **Helmet, CORS, express-rate-limit, mongo-sanitize, hpp**
* **Bcrypt** for password hashing

## Architecture style

Use a **modular monolith** first.

That means:

* one backend repo
* clean module separation
* each module has controller, service, model, validator, routes
* easy to break into microservices later if scale demands it

For your CRM, this is the right choice because:

* faster to build
* easier to deploy
* less ops complexity
* enough for production if coded cleanly

## Core architectural principles

### Multi-tenant by company

Every company-owned record should carry:

* `companyId`

So leads, deals, contacts, tasks, payments, tickets, settings, permissions, notifications, events all stay tenant-isolated.

### Platform-level vs company-level data

Separate clearly:

**Platform-level**

* plans
* companies
* platform analytics
* platform audit logs
* super admin users

**Company-level**

* employees/users
* leads
* deals
* contacts
* tasks
* payments
* tickets
* company settings
* permissions
* modules
* workflows

### Authorization layers

Access should be checked in this order:

1. Valid JWT?
2. User active?
3. Company active and subscription valid?
4. Module enabled for this company?
5. User role allowed to access module?
6. User action allowed? view/create/edit/delete
7. Record ownership scope applies? assigned/self/team/all

That is how you properly support your Dynamic Visibility Engine + Permission Matrix.

---

# 3) Production-ready folder structure

```bash
trackfield-crm-backend/
├── src/
│   ├── app.js
│   ├── server.js
│   │
│   ├── config/
│   │   ├── env.js
│   │   ├── db.js
│   │   ├── logger.js
│   │   ├── redis.js
│   │   └── constants.js
│   │
│   ├── modules/
│   │   ├── auth/
│   │   │   ├── auth.controller.js
│   │   │   ├── auth.service.js
│   │   │   ├── auth.routes.js
│   │   │   ├── auth.validation.js
│   │   │   └── auth.tokens.js
│   │   │
│   │   ├── users/
│   │   ├── companies/
│   │   ├── plans/
│   │   ├── roles/
│   │   ├── permissions/
│   │   ├── modules-control/
│   │   ├── dashboards/
│   │   ├── leads/
│   │   ├── deals/
│   │   ├── contacts/
│   │   ├── tasks/
│   │   ├── calendar/
│   │   ├── payments/
│   │   ├── tickets/
│   │   ├── notifications/
│   │   ├── automation/
│   │   ├── reports/
│   │   ├── settings/
│   │   └── activity-logs/
│   │
│   ├── models/
│   │   ├── User.js
│   │   ├── Company.js
│   │   ├── Plan.js
│   │   ├── CompanySubscription.js
│   │   ├── Role.js
│   │   ├── Permission.js
│   │   ├── CompanyModule.js
│   │   ├── Lead.js
│   │   ├── Deal.js
│   │   ├── Contact.js
│   │   ├── Task.js
│   │   ├── CalendarEvent.js
│   │   ├── Payment.js
│   │   ├── Ticket.js
│   │   ├── Notification.js
│   │   ├── Workflow.js
│   │   ├── ActivityLog.js
│   │   ├── CompanySetting.js
│   │   ├── RefreshToken.js
│   │   └── Otp.js
│   │
│   ├── middlewares/
│   │   ├── auth.middleware.js
│   │   ├── role.middleware.js
│   │   ├── permission.middleware.js
│   │   ├── module-access.middleware.js
│   │   ├── tenant.middleware.js
│   │   ├── validate.middleware.js
│   │   ├── error.middleware.js
│   │   ├── notFound.middleware.js
│   │   └── rateLimit.middleware.js
│   │
│   ├── utils/
│   │   ├── ApiError.js
│   │   ├── ApiResponse.js
│   │   ├── asyncHandler.js
│   │   ├── pagination.js
│   │   ├── queryBuilder.js
│   │   ├── jwt.js
│   │   ├── password.js
│   │   ├── dates.js
│   │   └── audit.js
│   │
│   ├── jobs/
│   │   ├── email.job.js
│   │   ├── notification.job.js
│   │   ├── workflow.job.js
│   │   └── payment-reminder.job.js
│   │
│   ├── events/
│   │   └── socket.js
│   │
│   └── routes/
│       ├── index.js
│       └── v1.js
│
├── tests/
├── .env
├── .env.example
├── package.json
└── README.md
```

## Why this structure works

* modules stay isolated
* models are reusable
* middleware is centralized
* future scaling is easy
* clean enough for team collaboration

---

# 4) MongoDB schema design

Below is the production-level schema design.

## A. User

```js
{
  _id,
  companyId: ObjectId | null, // null for super admin if platform user
  firstName: String,
  lastName: String,
  fullName: String,
  email: { type: String, unique: true, index: true },
  phone: String,
  passwordHash: String,
  avatar: String,

  roleKey: {
    type: String,
    enum: ['SUPER_ADMIN', 'ADMIN', 'HR_MANAGER', 'HR_EXECUTIVE', 'EMPLOYEE']
  },

  department: String,
  designation: String,

  employeeId: String,
  managerId: ObjectId,

  status: {
    type: String,
    enum: ['ACTIVE', 'INVITED', 'SUSPENDED', 'DISABLED'],
    default: 'ACTIVE'
  },

  isEmailVerified: { type: Boolean, default: false },
  lastLoginAt: Date,
  lastActiveAt: Date,

  permissionsVersion: Number, // helps invalidate stale permission cache
  createdBy: ObjectId,
  updatedBy: ObjectId
}
```

### Notes

* Global unique email is best for SaaS login
* `companyId = null` for top-level platform owner if needed
* `roleKey` is primary role, but granular permissions come separately

---

## B. Company

```js
{
  _id,
  name: String,
  slug: { type: String, unique: true, index: true },
  legalName: String,
  industry: String,
  companySize: String,
  website: String,
  email: String,
  phone: String,

  address: {
    line1: String,
    line2: String,
    city: String,
    state: String,
    country: String,
    postalCode: String
  },

  logo: String,
  timezone: String,
  currency: String,
  locale: String,

  status: {
    type: String,
    enum: ['ACTIVE', 'TRIAL', 'SUSPENDED', 'CANCELLED'],
    default: 'TRIAL'
  },

  ownerUserId: ObjectId,
  currentPlanId: ObjectId,

  trialEndsAt: Date,
  subscriptionEndsAt: Date,

  createdBy: ObjectId,
  updatedBy: ObjectId
}
```

---

## C. Plan

```js
{
  _id,
  name: String,
  code: { type: String, unique: true },
  priceMonthly: Number,
  priceYearly: Number,
  currency: String,
  features: [String],
  limits: {
    users: Number,
    leads: Number,
    deals: Number,
    contacts: Number,
    automations: Number,
    storageGb: Number
  },
  enabledModules: [String],
  isActive: { type: Boolean, default: true }
}
```

---

## D. CompanySubscription

```js
{
  _id,
  companyId: ObjectId,
  planId: ObjectId,
  billingCycle: { type: String, enum: ['MONTHLY', 'YEARLY'] },
  amount: Number,
  currency: String,
  status: { type: String, enum: ['ACTIVE', 'PAST_DUE', 'CANCELLED', 'TRIAL'] },
  startsAt: Date,
  endsAt: Date,
  autoRenew: Boolean,
  paymentProvider: String,
  externalSubscriptionId: String
}
```

---

## E. Role

Use company-specific roles with defaults.

```js
{
  _id,
  companyId: ObjectId, // null for system default roles
  name: String,        // e.g. HR Manager
  key: String,         // HR_MANAGER
  description: String,
  isSystem: Boolean,
  isActive: Boolean
}
```

---

## F. Permission

This is your permission matrix.

```js
{
  _id,
  companyId: ObjectId,
  roleId: ObjectId,
  moduleKey: String, // leads, deals, contacts, payments, etc.
  actions: {
    view: Boolean,
    create: Boolean,
    edit: Boolean,
    delete: Boolean,
    assign: Boolean,
    export: Boolean
  },
  scope: {
    type: String,
    enum: ['SELF', 'TEAM', 'ALL'],
    default: 'SELF'
  }
}
```

This schema directly supports the “per-role × per-module × per-action” behavior in your frontend. ([GitHub][1])

## G. CompanyModule

This is your Dynamic Visibility Engine.

```js
{
  _id,
  companyId: ObjectId,
  moduleKey: String, // leads, deals, payments...
  enabled: Boolean,
  customLabel: String,
  hiddenForRoles: [ObjectId], // optional
  settings: {
    allowExport: Boolean,
    allowImport: Boolean
  }
}
```

This supports the frontend’s module control where a company can hide modules like Payments or Leads from downstream users. ([GitHub][1])

## H. Lead

```js
{
  _id,
  companyId: ObjectId,
  leadId: String, // human-readable code
  firstName: String,
  lastName: String,
  email: String,
  phone: String,
  companyName: String,
  source: String,
  status: String,      // new, contacted, qualified, lost
  stage: String,
  priority: String,
  tags: [String],

  assignedTo: ObjectId,
  createdBy: ObjectId,
  updatedBy: ObjectId,

  notes: [
    {
      text: String,
      createdBy: ObjectId,
      createdAt: Date
    }
  ],

  expectedValue: Number,
  lastContactedAt: Date,
  nextFollowUpAt: Date
}
```

Indexes:

* `companyId + status`
* `companyId + assignedTo`
* `companyId + createdAt`
* text index on name/email/companyName

---

## I. Deal

```js
{
  _id,
  companyId: ObjectId,
  dealId: String,
  title: String,
  contactId: ObjectId,
  leadId: ObjectId,
  pipeline: String,
  stage: String,
  amount: Number,
  currency: String,
  probability: Number,
  expectedCloseDate: Date,

  assignedTo: ObjectId,
  createdBy: ObjectId,
  updatedBy: ObjectId,

  status: {
    type: String,
    enum: ['OPEN', 'WON', 'LOST'],
    default: 'OPEN'
  },

  notes: [
    {
      text: String,
      createdBy: ObjectId,
      createdAt: Date
    }
  ]
}
```

---

## J. Contact

```js
{
  _id,
  companyId: ObjectId,
  contactId: String,
  firstName: String,
  lastName: String,
  email: String,
  phone: String,
  alternatePhone: String,
  organization: String,
  designation: String,
  tags: [String],
  ownerId: ObjectId,
  address: {
    city: String,
    state: String,
    country: String
  },
  linkedLeadId: ObjectId,
  linkedDealIds: [ObjectId],
  createdBy: ObjectId,
  updatedBy: ObjectId
}
```

---

## K. Task

```js
{
  _id,
  companyId: ObjectId,
  title: String,
  description: String,
  moduleKey: String, // lead, deal, ticket, general
  relatedEntityId: ObjectId,
  assignedTo: ObjectId,
  assignedBy: ObjectId,
  priority: { type: String, enum: ['LOW', 'MEDIUM', 'HIGH', 'URGENT'] },
  status: { type: String, enum: ['TODO', 'IN_PROGRESS', 'DONE', 'CANCELLED'] },
  dueDate: Date,
  completedAt: Date
}
```

---

## L. CalendarEvent

```js
{
  _id,
  companyId: ObjectId,
  title: String,
  description: String,
  type: String, // meeting, follow-up, demo, reminder
  color: String,
  startAt: Date,
  endAt: Date,
  allDay: Boolean,
  attendees: [ObjectId],
  relatedEntityType: String,
  relatedEntityId: ObjectId,
  createdBy: ObjectId,
  updatedBy: ObjectId
}
```

This fits the calendar grid and event-dot behavior in your frontend. ([GitHub][1])

## M. Payment / Invoice

```js
{
  _id,
  companyId: ObjectId,
  invoiceNo: String,
  customerName: String,
  customerEmail: String,
  amount: Number,
  currency: String,
  status: { type: String, enum: ['PENDING', 'PAID', 'OVERDUE', 'CANCELLED'] },
  issueDate: Date,
  dueDate: Date,
  paidAt: Date,
  notes: String,
  createdBy: ObjectId,
  updatedBy: ObjectId
}
```

---

## N. Ticket

```js
{
  _id,
  companyId: ObjectId,
  ticketNo: String,
  subject: String,
  description: String,
  category: String,
  priority: { type: String, enum: ['LOW', 'MEDIUM', 'HIGH', 'CRITICAL'] },
  status: { type: String, enum: ['OPEN', 'IN_PROGRESS', 'RESOLVED', 'CLOSED'] },
  requesterId: ObjectId,
  assignedTo: ObjectId,
  comments: [
    {
      message: String,
      authorId: ObjectId,
      createdAt: Date
    }
  ],
  createdBy: ObjectId,
  updatedBy: ObjectId
}
```

---

## O. Notification

```js
{
  _id,
  companyId: ObjectId,
  userId: ObjectId,
  type: String, // lead_assigned, task_due, payment_overdue...
  title: String,
  message: String,
  data: Object,
  isRead: { type: Boolean, default: false },
  readAt: Date
}
```

This supports badge count, dismiss, and mark-all-read. ([GitHub][1])

## P. Workflow / Automation

```js
{
  _id,
  companyId: ObjectId,
  name: String,
  description: String,
  moduleKey: String,
  trigger: {
    event: String, // lead.created, deal.stage_changed, task.overdue
    conditions: [Object]
  },
  actions: [
    {
      type: String, // assign_user, create_task, send_notification, update_status
      config: Object
    }
  ],
  isActive: Boolean,
  createdBy: ObjectId,
  updatedBy: ObjectId
}
```

---

## Q. CompanySetting

```js
{
  _id,
  companyId: ObjectId,
  profile: {
    companyName: String,
    email: String,
    phone: String,
    website: String,
    address: Object
  },
  security: {
    passwordPolicy: String,
    sessionTimeoutMins: Number,
    twoFactorEnabled: Boolean
  },
  notifications: {
    email: Boolean,
    push: Boolean,
    taskReminders: Boolean,
    paymentAlerts: Boolean
  },
  branding: {
    primaryColor: String,
    logo: String
  }
}
```

---

## R. ActivityLog

```js
{
  _id,
  companyId: ObjectId | null,
  actorId: ObjectId,
  actorRole: String,
  moduleKey: String,
  action: String, // create/update/delete/login/export
  entityType: String,
  entityId: ObjectId,
  description: String,
  metadata: Object,
  ipAddress: String,
  userAgent: String,
  createdAt: Date
}
```

This supports both platform-wide and company-specific audit trails. ([GitHub][1])

## S. RefreshToken

```js
{
  _id,
  userId: ObjectId,
  tokenHash: String,
  expiresAt: Date,
  revokedAt: Date,
  deviceInfo: String,
  ipAddress: String
}
```

---

# 5) REST API structure

Use versioned APIs:

```bash
/api/v1
```

## Auth

```bash
POST   /auth/login
POST   /auth/refresh-token
POST   /auth/logout
POST   /auth/forgot-password
POST   /auth/reset-password
GET    /auth/me
```

## Session bootstrap

Very useful for frontend after login.

```bash
GET    /bootstrap
```

Response should include:

* user
* company
* role
* enabledModules
* permissions
* notificationCount

## Companies

```bash
GET    /companies
POST   /companies
GET    /companies/:id
PATCH  /companies/:id
DELETE /companies/:id
PATCH  /companies/:id/status
GET    /companies/:id/analytics
```

## Plans / subscriptions

```bash
GET    /plans
POST   /plans
PATCH  /plans/:id
GET    /subscriptions
POST   /subscriptions
PATCH  /subscriptions/:id
```

## Users / employees

```bash
GET    /users
POST   /users
GET    /users/:id
PATCH  /users/:id
DELETE /users/:id
PATCH  /users/:id/status
GET    /users/:id/permissions
PATCH  /users/:id/permissions
```

You can alias `/employees` to `/users` internally for company-facing routes:

```bash
GET    /employees
POST   /employees
...
```

## Roles & permissions

```bash
GET    /roles
POST   /roles
PATCH  /roles/:id
DELETE /roles/:id

GET    /permissions
PUT    /permissions/role/:roleId
```

## Module control

```bash
GET    /modules
PUT    /modules
PATCH  /modules/:moduleKey
```

Example:

* enable/disable leads
* rename module label
* hide for specific role

## Leads

```bash
GET    /leads
POST   /leads
GET    /leads/:id
PATCH  /leads/:id
DELETE /leads/:id
PATCH  /leads/:id/assign
PATCH  /leads/:id/status
POST   /leads/:id/notes
GET    /leads/kanban
```

## Deals

```bash
GET    /deals
POST   /deals
GET    /deals/:id
PATCH  /deals/:id
DELETE /deals/:id
PATCH  /deals/:id/stage
PATCH  /deals/:id/assign
POST   /deals/:id/notes
GET    /deals/pipeline
```

## Contacts

```bash
GET    /contacts
POST   /contacts
GET    /contacts/:id
PATCH  /contacts/:id
DELETE /contacts/:id
GET    /contacts/tags
```

## Tasks

```bash
GET    /tasks
POST   /tasks
GET    /tasks/:id
PATCH  /tasks/:id
DELETE /tasks/:id
PATCH  /tasks/:id/status
```

## Calendar

```bash
GET    /events
POST   /events
GET    /events/:id
PATCH  /events/:id
DELETE /events/:id
GET    /events/month/:year/:month
GET    /events/upcoming
```

## Payments

```bash
GET    /payments
POST   /payments
GET    /payments/:id
PATCH  /payments/:id
DELETE /payments/:id
PATCH  /payments/:id/status
```

## Tickets

```bash
GET    /tickets
POST   /tickets
GET    /tickets/:id
PATCH  /tickets/:id
PATCH  /tickets/:id/status
POST   /tickets/:id/comments
```

## Notifications

```bash
GET    /notifications
PATCH  /notifications/:id/read
PATCH  /notifications/read-all
DELETE /notifications/:id
GET    /notifications/unread-count
```

## Automation

```bash
GET    /workflows
POST   /workflows
GET    /workflows/:id
PATCH  /workflows/:id
DELETE /workflows/:id
PATCH  /workflows/:id/toggle
```

## Reports / dashboards

```bash
GET    /dashboard/super-admin
GET    /dashboard/company
GET    /dashboard/user

GET    /reports/leads
GET    /reports/deals
GET    /reports/payments
GET    /reports/team-performance
```

## Settings

```bash
GET    /settings/company
PATCH  /settings/company
PATCH  /settings/security
PATCH  /settings/notifications
```

## Activity logs

```bash
GET    /activity-logs
GET    /activity-logs/:id
```

---

# 6) Recommended route split by role

## Super Admin can access

* auth
* bootstrap
* companies
* plans
* subscriptions
* global users
* platform analytics
* platform activity logs

## Company Admin can access

* auth
* bootstrap
* company users/employees
* roles
* permissions
* modules
* leads
* deals
* contacts
* tasks
* events
* payments
* tickets
* workflows
* reports
* settings
* company activity logs

## Employee can access

* auth
* bootstrap
* my dashboard
* filtered leads/deals/tasks/contacts/tickets/events/notifications
* only based on permission + scope

---

# 7) Best practices: auth, RBAC, validation, security

## JWT strategy

Use:

* **short-lived access token**: 15 min
* **refresh token**: 7–30 days

### Access token payload

```js
{
  sub: userId,
  companyId,
  roleKey,
  sessionId
}
```

### Refresh token

* store hashed token in DB
* rotate on refresh
* revoke on logout

## Recommended middleware chain

For protected route:

```js
router.get(
  "/leads",
  authenticate,
  checkCompanyActive,
  checkModuleAccess("leads"),
  checkPermission("leads", "view"),
  leadsController.list
);
```

## RBAC + ABAC

Use both:

* **RBAC**: role based
* **ABAC/scope**: self/team/all

Example:

* HR Executive may view leads but only assigned ones
* Admin may view all company leads

## Validation

Use Zod or Joi on:

* params
* query
* body

Example:

* email format
* Mongo ObjectId
* enum status values
* pagination limits
* no negative amounts

## Security checklist

Use:

* `helmet()`
* strict CORS whitelist
* rate limit on auth routes
* request size limits
* password hashing with bcrypt
* JWT secret in env
* sanitize Mongo operators
* `hpp()` to prevent HTTP param pollution
* secure cookies if using cookie-based refresh token
* audit logs for sensitive actions
* do not return raw stack traces in production

## Sensitive action auditing

Log these:

* login
* failed login
* create/update/delete
* permission change
* module enable/disable
* user suspension
* plan change
* export action

## Pagination/filter/sort

Every list API should support:

```bash
?page=1&limit=20&search=rohan&sortBy=createdAt&sortOrder=desc
```

## Soft delete

For important business records, prefer:

* `deletedAt`
* `deletedBy`

rather than hard delete

## Indexing

You should add indexes on:

* `companyId`
* `assignedTo`
* `email`
* `status`
* `createdAt`
* compound indexes for high-use filters

---

# 8) Suggested response format for frontend integration

Keep your API responses consistent.

## Success

```json
{
  "success": true,
  "message": "Leads fetched successfully",
  "data": [...],
  "meta": {
    "page": 1,
    "limit": 20,
    "total": 140
  }
}
```

## Error

```json
{
  "success": false,
  "message": "Unauthorized",
  "errors": []
}
```

---

# 9) How your frontend modules map to backend modules

## Super Admin dashboard

Needs:

* company count
* active subscriptions
* total users
* MRR/ARR
* industry distribution
* recent activity

## Company dashboard

Needs:

* total employees
* leads by stage
* deals by pipeline stage
* tasks due soon
* payments due/pending
* team performance summary

## User dashboard

Needs:

* my tasks
* my leads
* my deals
* tickets assigned to me
* events upcoming
* notifications

That aligns cleanly with the three frontend dashboard layers documented in the repo. ([GitHub][1])

# 10) Suggested module execution order

Do not build everything at once.

## Phase 1

* auth
* companies
* users/employees
* roles/permissions
* module control
* leads
* bootstrap

## Phase 2

* deals
* contacts
* tasks
* notifications
* dashboard

## Phase 3

* calendar
* payments
* tickets
* reports
* activity logs

## Phase 4

* automation
* subscription billing
* websockets/realtime
* queues

---

# 11) Recommended Mongoose relations summary

* Company has many Users
* Company has one active Subscription
* Company has many Roles
* Role has many Permissions
* Company has many Module configs
* Company has many Leads, Deals, Contacts, Tasks, Events, Payments, Tickets
* User can be assigned many Leads, Deals, Tasks, Tickets
* ActivityLog references actor and entity
* Notifications belong to user

---

# 12) One practical improvement for your current frontend integration

Instead of making the frontend separately call:

* user
* company
* permissions
* modules
* unread notifications

Create one **bootstrap endpoint**:

```bash
GET /api/v1/bootstrap
```

Response:

```json
{
  "user": {},
  "company": {},
  "role": {},
  "permissions": [],
  "enabledModules": [],
  "notificationCount": 4
}
```

This makes your React app faster and cleaner on first load.

---

# 13) Final recommendation

For **TrackField CRM**, the best production path is:

* **Node.js + Express modular monolith**
* **MongoDB with strict company-based tenant isolation**
* **JWT access + refresh tokens**
* **permission matrix stored in DB**
* **module control stored in DB**
* **service-layer architecture**
* **Redis + queues for async jobs**
* **audit logging for sensitive changes**
* **one bootstrap endpoint for frontend hydration**

This architecture matches your current frontend’s three-level SaaS model, dynamic visibility engine, role-permission matrix, notification system, and calendar/task flows already documented in the repo. ([GitHub][1])an turn this into the next step as a complete backend starter template with actual Express folder code, Mongoose models, JWT middleware, and sample routes.

[1]: https://github.com/niveshbansal07/CRM/tree/main/Frontend "CRM/Frontend at main · niveshbansal07/CRM · GitHub"

